##Load Data


In [ ]:
import pandas as pd

In [ ]:
df= pd.read_csv("/content/gk_qna_dataset.csv")
df.head()

In [ ]:
df['question'][0]

## Tokenization

In [ ]:
import re
def tokenize(text):
  #lowercase
  text=text.lower()
  #remove quotes
  text=re.sub(r"[\"']","",text)
  #remove puntuation
  text=re.sub(r"[^a-z0-9\s]","  ",text)

  #remove extra space
  text=re.sub(r"\s+"," ",text).strip()

  return text.split()

In [ ]:
print(tokenize(df['question'][0]))

##Vocab Forming

In [ ]:
vocab={'<unk>': 0}

In [ ]:
def build_vocab(row):
  #print(row['question'],row['answer'])
  tokenized_question=tokenize(row['question'])
  tokenized_answer=tokenize(row['answer'])
  #print(tokenized_question,tokenized_answer)
  merged_tokens=tokenized_question+tokenized_answer
  #print(merged_tokens)
  for token in merged_tokens:
    if token not in vocab:
      vocab[token]=len(vocab)


In [ ]:
df.apply(build_vocab,axis=1)

In [ ]:
vocab

In [ ]:
len(vocab)

## Text to indices

In [ ]:
def text_to_indices(text,vocab):
  indexed_text=[]
  for token in tokenize(text):
    if token not in vocab:
       #print(vocab['<unk>'])
       indexed_text.append(vocab['<unk>'])
    else:
      #print(vocab[token])
      indexed_text.append(vocab[token])
  return indexed_text

In [ ]:
text_to_indices('what is your name',vocab)

## Dataset and Dataloader

In [ ]:
import torch
from torch.utils.data import Dataset,DataLoader

In [ ]:
len(df)

In [ ]:
class QADataset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab

  def __len__(self):
    return len(self.df)

  def __getitem__(self,idx):
    question=text_to_indices(self.df.iloc[idx]['question'],self.vocab)
    answer=text_to_indices(self.df.iloc[idx]['answer'],self.vocab)
    return torch.tensor(question),torch.tensor(answer)

In [ ]:
dataset=QADataset(df,vocab)

In [ ]:
dataset[1]

In [ ]:
len(dataset)

In [ ]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)
for question,answer in dataloader:
  print(question,answer[0])
  break

##RNN Architecture

In [ ]:
import torch.nn as nn
class simpleRNN(nn.Module):
  def __init__(self,vocab_size,embedding_dim=50,hidden_size=64):
    super().__init__()

    self.embedding=nn.Embedding(vocab_size,embedding_dim)
    self.rnn= nn.RNN(embedding_dim,hidden_size,batch_first=True)
    self.fc= nn.Linear(hidden_size,vocab_size)

  def forward(self,question):
    embedded=self.embedding(question)#(1,seq_len,50)
    _,final=self.rnn(embedded)
    return self.fc(final.squeeze(0))#(1,vocab_size)

In [ ]:
model=simpleRNN(vocab_size=len(vocab))

## Training Loop

In [ ]:
loss=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
epochs=50

In [ ]:
for epoch in range(epochs):
  total_loss=0
  for question,answer in dataloader:
    optimizer.zero_grad()
    output=model(question)#(1,vocab_size)

    target=answer[0][0].unsqueeze(0)

    loss_=loss(output,target)
    loss_.backward()
    optimizer.step()
    total_loss+=loss_.item()

  if (epoch+1)%10==0:
    print(f"Epoch:{epoch+1:3d}/{epochs} Loss:{total_loss:.4f}")



## Prediction

In [ ]:
vocab.items()

In [ ]:
idx_to_word={v:k for k,v in vocab.items()}
idx_to_word

In [ ]:
def predict(question_text,model,vocab,idx_to_word):
  model.eval()

  with torch.no_grad():
    indices=text_to_indices(question_text,vocab)
    if not indices:
      return '<UNK>'

    x=torch.tensor(indices).unsqueeze(0)
    pred=torch.argmax(model(x),dim=1).item()
  return idx_to_word.get(pred,'<UNK>')

In [31]:
tests=[
    'What is the capital of France?',
    'what is H20 commonly called?',
    'Which planet is know as the red planet?',
    'Which bird cannot fly?',
    'What is the capital of Japan?'
]
for question in tests:
  print(f"{question:45} -> {predict(question,model,vocab,idx_to_word)}")

What is the capital of France?                -> paris
what is H20 commonly called?                  -> water
Which planet is know as the red planet?       -> mars
Which bird cannot fly?                        -> ostrich
What is the capital of Japan?                 -> tokyo
